# BEAM Linkstats Consist Analysis

This notebook bootstraps a Consist tracker against an **archived PILATES run** (scratch storage), lists linkstats artifacts by facets, and computes first-pass summary + delta metrics over phys-sim iterations.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

print("Kernel Python:", sys.executable)


In [ ]:
# --- Configure these for your environment/run ---
USER = os.environ.get("USER", "$USER")
RUN_NAME = "new-asim-consist-flash-20260209-103617"
SLURM_JOB_ID = os.environ.get("SLURM_JOB_ID", "")

OUTPUT_ROOT = Path(f"/global/scratch/users/{USER}/pilates-output").resolve()
PROJECT_ROOT = Path(f"/global/scratch/users/{USER}/sources/PILATES").resolve()
ARCHIVE_RUN_DIR = OUTPUT_ROOT / RUN_NAME

# DB path resolution order:
# 1) PILATES_DB_PATH env override
# 2) node-local default (/local/job$SLURM_JOB_ID/db/pilates-analysis.duckdb)
# 3) archived run DB
DB_PATH_ENV = os.environ.get("PILATES_DB_PATH")
LOCAL_DB_PATH = Path(f"/local/job{SLURM_JOB_ID}/db/pilates-analysis.duckdb") if SLURM_JOB_ID else None
ARCHIVE_DB_PATH = ARCHIVE_RUN_DIR / "provenance.duckdb"

if DB_PATH_ENV:
    DB_PATH = Path(DB_PATH_ENV).expanduser()
elif LOCAL_DB_PATH is not None and LOCAL_DB_PATH.exists():
    DB_PATH = LOCAL_DB_PATH
else:
    DB_PATH = ARCHIVE_DB_PATH

TARGET_YEAR = 2018

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARCHIVE_RUN_DIR:", ARCHIVE_RUN_DIR)
print("LOCAL_DB_PATH:", LOCAL_DB_PATH)
print("DB_PATH:", DB_PATH)


In [ ]:
from pilates.utils.consist_analysis import (
    assign_effective_beam_sub_iteration,
    build_linkstats_hourly_pca_matrix,
    create_analysis_tracker,
    find_linkstats_artifacts,
    print_duckdb_health,
    summarize_linkstats_artifacts,
    summarize_linkstats_traveltime_deltas,
    summarize_linkstats_traveltime_deltas_hourly_weighted,
)

db_health = print_duckdb_health(db_path=DB_PATH, probe_open=True)

tracker = create_analysis_tracker(
    db_path=DB_PATH,
    archive_run_dir=ARCHIVE_RUN_DIR,
    project_root=PROJECT_ROOT,
    output_root=OUTPUT_ROOT,
)

print("Tracker mounts:")
for name, root in tracker.mounts.items():
    print(f"  {name}:// -> {root}")


In [ ]:
# Primary query: phys-sim unmodified linkstats parquet artifacts for one year across all outer iterations
artifacts_df = find_linkstats_artifacts(
    tracker,
    year=TARGET_YEAR,
    artifact_family="linkstats_unmodified_phys_sim_iter_parquet",
    namespace="beam",
    limit=20000,
)

print(f"Found {len(artifacts_df)} artifacts")
artifacts_df.head(20)

In [ ]:
# Quick facet/view sanity check
if artifacts_df.empty:
    print("No linkstats artifacts found with the requested filters.")
else:
    display(
        artifacts_df[[
            "key",
            "container_uri",
            "parent_run_id",
            "header_run_id",
            "facet_source",
            "phys_sim_iteration",
            "beam_sub_iteration",
        ]].head(25)
    )
    print("Facet source summary:")
    display(artifacts_df["facet_source"].value_counts(dropna=False))


In [ ]:
# Keep only indexed snapshots and derive an effective sub-iteration index.
# Promoted final sub-iteration rows (beam_sub_iteration = NaN) are mapped to
# max(non-null sub-iteration) + 1 within each (run_id, year, iteration, phys_sim_iteration).
artifacts_clean = artifacts_df[artifacts_df["facet_source"] == "artifact_kv"].copy()
artifacts_clean = assign_effective_beam_sub_iteration(artifacts_clean)

# Keep raw value for debugging, but use effective value for analysis grouping.
artifacts_clean["beam_sub_iteration_raw"] = artifacts_clean["beam_sub_iteration"]
artifacts_clean["beam_sub_iteration"] = artifacts_clean["beam_sub_iteration_effective"]

# Optional logical de-dup in case multiple keys map to the same snapshot.
artifacts_clean = (
    artifacts_clean
    .sort_values(["run_id", "year", "iteration", "phys_sim_iteration", "beam_sub_iteration", "key"])
    .drop_duplicates(
        subset=["run_id", "year", "iteration", "phys_sim_iteration", "beam_sub_iteration"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(f"Raw artifacts: {len(artifacts_df)}")
print(f"Filtered artifacts: {len(artifacts_clean)}")

# Use volume-weighted travel time in per-artifact summary stats.
summary_df = summarize_linkstats_artifacts(
    artifacts_clean,
    tracker=tracker,
    traveltime_weighting="volume_weighted",
)
print(f"Computed summaries for {len(summary_df)} artifacts via Consist views")
summary_df.head(20)


In [ ]:
# First-pass rollup by sub-iteration + phys-sim iteration
if summary_df.empty:
    print("No summary rows available. Check mounts/path resolution above.")
else:
    rollup = (
        summary_df.groupby(["parent_run_id", "header_run_id", "year", "iteration", "beam_sub_iteration", "phys_sim_iteration"], dropna=False)
        .agg(
            artifacts=("key", "count"),
            rows_total=("row_count", "sum"),
            distinct_links_mean=("distinct_links", "mean"),
            volume_sum_total=("volume_sum", "sum"),
            vmt_miles_total=("vmt_miles", "sum"),
            vht_hours_total=("vht_hours", "sum"),
            total_delay_hours_total=("total_delay_hours", "sum"),
            vmt_per_vht_mph_mean=("vmt_per_vht_mph", "mean"),
            vht_per_vmt_hours_per_mile_mean=("vht_per_vmt_hours_per_mile", "mean"),
            traveltime_mean_mean=("traveltime_mean", "mean"),
            traveltime_p95_mean=("traveltime_p95", "mean"),
        )
        .assign(
            vmt_per_vht_mph_total=lambda df: df["vmt_miles_total"] / df["vht_hours_total"],
            vht_per_vmt_hours_per_mile_total=lambda df: df["vht_hours_total"] / df["vmt_miles_total"],
        )
        .reset_index()
        .sort_values(["parent_run_id", "header_run_id", "year", "iteration", "beam_sub_iteration", "phys_sim_iteration"], na_position="last")
    )
    display(rollup)


In [ ]:
# Consecutive phys-sim travel-time deltas within each (year, iteration, beam_sub_iteration) sequence
# Set USE_HOURLY_WEIGHTED = True to run the faster, hourly volume-weighted variant.
USE_HOURLY_WEIGHTED = True

if USE_HOURLY_WEIGHTED:
    delta_df = summarize_linkstats_traveltime_deltas_hourly_weighted(
        summary_df,
        tracker=tracker,
        exclude_zero_volume=True,
    )
    print(f"Computed {len(delta_df)} hourly-weighted travel-time delta rows")
else:
    delta_df = summarize_linkstats_traveltime_deltas(summary_df, tracker=tracker)
    print(f"Computed {len(delta_df)} travel-time delta rows")

delta_df


In [ ]:
# Plots
if delta_df.empty:
    print("delta_df is empty; skipping delta plots.")
else:
    sns.relplot(
        data=delta_df,
        x="phys_sim_iteration_prev",
        y="traveltime_delta_abs_mean",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )

if "rollup" in globals() and not rollup.empty:
    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="traveltime_mean_mean",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
    )

    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="total_delay_hours_total",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
    )

    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="vmt_per_vht_mph_total",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
    )

    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="vmt_miles_total",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )

    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=rollup,
        x="vmt_miles_total",
        y="traveltime_mean_mean",
        hue="beam_sub_iteration",
        style="iteration",
        s=70,
    )
    plt.title("Travel Time vs VMT by Iteration/Sub-Iteration")
    plt.tight_layout()

if not delta_df.empty and {"prev_volume_total", "curr_volume_total"}.issubset(delta_df.columns):
    melt_df = delta_df.melt(
        id_vars=["iteration", "beam_sub_iteration", "phys_sim_iteration_prev"],
        value_vars=["prev_volume_total", "curr_volume_total"],
        var_name="period",
        value_name="volume_total",
    )
    sns.relplot(
        data=melt_df,
        x="phys_sim_iteration_prev",
        y="volume_total",
        hue="period",
        col="beam_sub_iteration",
        row="iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )


In [ ]:
# Multi-scenario faceted plots (by parent/header run)
if "summary_df" not in globals() or summary_df.empty:
    print("summary_df not available; run artifact summary first.")
else:
    run_map = (
        summary_df[["artifact_id", "parent_run_id", "header_run_id"]]
        .drop_duplicates(subset=["artifact_id"], keep="first")
        .copy()
    )
    run_map["parent_run_id"] = run_map["parent_run_id"].fillna("<none>").astype(str)
    run_map["header_run_id"] = run_map["header_run_id"].fillna(run_map["parent_run_id"]).astype(str)

    scenario_count = run_map["parent_run_id"].nunique(dropna=False)
    print(f"Detected {scenario_count} parent/header run group(s).")

    if "rollup" in globals() and not rollup.empty and "parent_run_id" in rollup.columns:
        rollup_plot = rollup.copy()
        rollup_plot["parent_run_id"] = rollup_plot["parent_run_id"].fillna("<none>").astype(str)

        sns.relplot(
            data=rollup_plot,
            x="phys_sim_iteration",
            y="traveltime_mean_mean",
            row="parent_run_id" if scenario_count > 1 else None,
            col="iteration",
            hue="beam_sub_iteration",
            kind="line",
            marker="o",
            facet_kws={"sharey": False},
        )

        if "total_delay_hours_total" in rollup_plot.columns:
            sns.relplot(
                data=rollup_plot,
                x="phys_sim_iteration",
                y="total_delay_hours_total",
                row="parent_run_id" if scenario_count > 1 else None,
                col="iteration",
                hue="beam_sub_iteration",
                kind="line",
                marker="o",
                facet_kws={"sharey": False},
            )

    if "delta_df" in globals() and not delta_df.empty:
        delta_plot = delta_df.copy()
        if "artifact_id_curr" in delta_plot.columns:
            delta_plot = delta_plot.merge(
                run_map.rename(columns={"artifact_id": "artifact_id_curr"}),
                on="artifact_id_curr",
                how="left",
            )
        if "parent_run_id" in delta_plot.columns:
            delta_plot["parent_run_id"] = delta_plot["parent_run_id"].fillna("<none>").astype(str)

        if "parent_run_id" in delta_plot.columns:
            sns.relplot(
                data=delta_plot,
                x="phys_sim_iteration_prev",
                y="traveltime_delta_abs_mean",
                row="parent_run_id" if scenario_count > 1 else None,
                col="iteration",
                hue="beam_sub_iteration",
                kind="line",
                marker="o",
                facet_kws={"sharey": False},
            )

    if "pca_scores_df" in globals() and not pca_scores_df.empty and {"PC1", "PC2"}.issubset(pca_scores_df.columns):
        pca_plot = pca_scores_df.copy()
        if "parent_run_id" in pca_plot.columns:
            pca_plot["parent_run_id"] = pca_plot["parent_run_id"].fillna("<none>").astype(str)

        sns.relplot(
            data=pca_plot,
            x="PC1",
            y="PC2",
            row="parent_run_id" if (scenario_count > 1 and "parent_run_id" in pca_plot.columns) else None,
            col="iteration" if "iteration" in pca_plot.columns else None,
            hue="beam_sub_iteration" if "beam_sub_iteration" in pca_plot.columns else None,
            kind="scatter",
            facet_kws={"sharex": False, "sharey": False},
        )


In [ ]:
# PC1-PC2 velocity vectors across consecutive phys-sim iterations
if "pca_scores_df" not in globals() or pca_scores_df.empty:
    print("pca_scores_df not available; run PCA cells first.")
elif not {"PC1", "PC2", "phys_sim_iteration"}.issubset(pca_scores_df.columns):
    print("Need PC1, PC2, phys_sim_iteration columns in pca_scores_df.")
else:
    vec_df = pca_scores_df.copy()
    for col in ["parent_run_id", "header_run_id", "iteration", "beam_sub_iteration"]:
        if col not in vec_df.columns:
            vec_df[col] = "<none>"

    vec_df["parent_run_id"] = vec_df["parent_run_id"].fillna("<none>").astype(str)
    vec_df["iteration"] = pd.to_numeric(vec_df["iteration"], errors="coerce")
    vec_df["beam_sub_iteration"] = pd.to_numeric(vec_df["beam_sub_iteration"], errors="coerce")
    vec_df["phys_sim_iteration"] = pd.to_numeric(vec_df["phys_sim_iteration"], errors="coerce")

    vec_df = vec_df.sort_values(
        ["parent_run_id", "iteration", "beam_sub_iteration", "phys_sim_iteration"],
        na_position="last",
    ).reset_index(drop=True)

    group_cols = ["parent_run_id", "iteration", "beam_sub_iteration"]
    vec_df["PC1_next"] = vec_df.groupby(group_cols, dropna=False)["PC1"].shift(-1)
    vec_df["PC2_next"] = vec_df.groupby(group_cols, dropna=False)["PC2"].shift(-1)
    vec_df["phys_sim_next"] = vec_df.groupby(group_cols, dropna=False)["phys_sim_iteration"].shift(-1)

    arrow_df = vec_df.dropna(subset=["PC1_next", "PC2_next", "phys_sim_next"]).copy()
    arrow_df = arrow_df[arrow_df["phys_sim_next"] == arrow_df["phys_sim_iteration"] + 1].copy()
    arrow_df["dPC1"] = arrow_df["PC1_next"] - arrow_df["PC1"]
    arrow_df["dPC2"] = arrow_df["PC2_next"] - arrow_df["PC2"]
    arrow_df["step_mag"] = np.sqrt(arrow_df["dPC1"] ** 2 + arrow_df["dPC2"] ** 2)

    print(f"Arrow segments: {len(arrow_df)}")
    if arrow_df.empty:
        print("No consecutive phys-sim pairs found for vector plotting.")
    else:
        display(
            arrow_df[
                [
                    "parent_run_id",
                    "iteration",
                    "beam_sub_iteration",
                    "phys_sim_iteration",
                    "phys_sim_next",
                    "dPC1",
                    "dPC2",
                    "step_mag",
                ]
            ].head(20)
        )

        # Single-panel overlay: scores and vectors in the same PC1-PC2 space.
        plt.figure(figsize=(8.5, 6.5))
        sns.scatterplot(
            data=vec_df,
            x="PC1",
            y="PC2",
            hue="iteration",
            style="beam_sub_iteration",
            s=50,
            alpha=0.75,
        )
        plt.quiver(
            arrow_df["PC1"].to_numpy(),
            arrow_df["PC2"].to_numpy(),
            arrow_df["dPC1"].to_numpy(),
            arrow_df["dPC2"].to_numpy(),
            angles="xy",
            scale_units="xy",
            scale=1,
            width=0.0022,
            alpha=0.55,
            color="black",
            zorder=2,
        )
        plt.title("Weighted PCA Scores with Consecutive Phys-Sim Velocity Vectors")
        plt.tight_layout()

        # Faceted vector fields (scenario x outer iteration).
        row_key = "parent_run_id" if vec_df["parent_run_id"].nunique(dropna=False) > 1 else None
        g = sns.FacetGrid(
            vec_df,
            row=row_key,
            col="iteration",
            hue="beam_sub_iteration",
            margin_titles=True,
            sharex=False,
            sharey=False,
            height=3.2,
            aspect=1.1,
        )
        g.map_dataframe(sns.scatterplot, x="PC1", y="PC2", s=35, alpha=0.80)

        row_names = g.row_names if row_key is not None else [None]
        col_names = g.col_names
        axes = np.atleast_2d(g.axes)

        for r_idx, row_name in enumerate(row_names):
            for c_idx, col_name in enumerate(col_names):
                ax = axes[r_idx, c_idx]
                mask = arrow_df["iteration"] == col_name
                if row_key is not None:
                    mask &= arrow_df[row_key] == row_name
                d = arrow_df[mask]
                if d.empty:
                    continue
                ax.quiver(
                    d["PC1"].to_numpy(),
                    d["PC2"].to_numpy(),
                    d["dPC1"].to_numpy(),
                    d["dPC2"].to_numpy(),
                    angles="xy",
                    scale_units="xy",
                    scale=1,
                    width=0.003,
                    alpha=0.55,
                    color="black",
                    zorder=2,
                )

        g.add_legend(title="beam_sub_iteration")
        g.fig.subplots_adjust(top=0.90)
        g.fig.suptitle("PC1-PC2 Velocity Field by Scenario and Iteration")

        # Coherence proxy: cosine similarity of each arrow to the mean direction per facet.
        coherence_rows = []
        for keys, group in arrow_df.groupby(["parent_run_id", "iteration"], dropna=False):
            mean_vec = group[["dPC1", "dPC2"]].mean().to_numpy()
            mean_norm = float(np.linalg.norm(mean_vec))
            if mean_norm == 0:
                continue
            vecs = group[["dPC1", "dPC2"]].to_numpy()
            vec_norm = np.linalg.norm(vecs, axis=1)
            valid = vec_norm > 0
            if not np.any(valid):
                continue
            cos_sim = (vecs[valid] @ mean_vec) / (vec_norm[valid] * mean_norm)
            coherence_rows.append(
                {
                    "parent_run_id": keys[0],
                    "iteration": keys[1],
                    "n_vectors": int(valid.sum()),
                    "mean_cosine_to_mean_direction": float(np.mean(cos_sim)),
                }
            )

        if coherence_rows:
            coherence_df = pd.DataFrame(coherence_rows).sort_values(
                ["parent_run_id", "iteration"], na_position="last"
            )
            display(coherence_df)


In [ ]:
# Build PCA matrix: rows=artifacts, cols=(selected links x hourly bins)
PCA_LINK_FILTER_STRATEGY = "all"  # "first" | "last" | "all"
PCA_VOLUME_COVERAGE = 0.98
PCA_HOUR_GROUP_SIZE = 1  # 1 => 24 bins
PCA_METRIC = "speed_mph"
PCA_EXCLUDE_ZERO_VOLUME = True

# DuckDB runtime tuning (helps parallelize heavy grouping scans on HPC)
PCA_DUCKDB_THREADS = max(1, (os.cpu_count() or 1) - 1)
PCA_DUCKDB_MEMORY_LIMIT = os.environ.get("PILATES_DUCKDB_MEMORY_LIMIT", "80GB")
PCA_DUCKDB_TEMP_DIR = f"/local/job{SLURM_JOB_ID}/duckdb-tmp" if SLURM_JOB_ID else None

pca_payload = build_linkstats_hourly_pca_matrix(
    summary_df,
    tracker=tracker,
    link_filter_strategy=PCA_LINK_FILTER_STRATEGY,
    volume_coverage=PCA_VOLUME_COVERAGE,
    hour_group_size=PCA_HOUR_GROUP_SIZE,
    metric=PCA_METRIC,
    exclude_zero_volume=PCA_EXCLUDE_ZERO_VOLUME,
    dtype="float32",
    duckdb_threads=PCA_DUCKDB_THREADS,
    duckdb_memory_limit=PCA_DUCKDB_MEMORY_LIMIT,
    duckdb_temp_directory=PCA_DUCKDB_TEMP_DIR,
    duckdb_preserve_insertion_order=False,
)

X = pca_payload["matrix"]
artifact_index_df = pca_payload["artifact_index"]
feature_index_df = pca_payload["feature_index"]

print("PCA matrix shape:", X.shape)
print("Artifacts:", len(artifact_index_df))
print("Selected links:", len(pca_payload["link_index"]))
print("Hour bins:", pca_payload["hour_bins"])
print("DuckDB threads:", PCA_DUCKDB_THREADS)
print("DuckDB memory limit:", PCA_DUCKDB_MEMORY_LIMIT)
print("DuckDB temp dir:", PCA_DUCKDB_TEMP_DIR)
print("Exclude zero volume:", PCA_EXCLUDE_ZERO_VOLUME)

display(artifact_index_df.head(10))
display(feature_index_df.head(10))


In [ ]:
# Column-volume-weighted PCA using NumPy SVD
if X.size == 0 or X.shape[0] < 2 or X.shape[1] == 0:
    print("Not enough data to run PCA.")
else:
    X64 = X.astype(np.float64, copy=False)
    X_centered = X64 - X64.mean(axis=0, keepdims=True)

    w = pca_payload["column_weights_normalized"].astype(np.float64, copy=False)
    if w.size != X_centered.shape[1]:
        raise ValueError("Column-weight length does not match feature count")

    sqrt_w = np.sqrt(np.clip(w, 0.0, None))
    X_weighted = X_centered * sqrt_w[None, :]

    U, S, Vt = np.linalg.svd(X_weighted, full_matrices=False)
    denom = max(X_weighted.shape[0] - 1, 1)
    explained_var = (S ** 2) / denom
    total_var = explained_var.sum()
    explained_ratio = explained_var / total_var if total_var > 0 else np.zeros_like(explained_var)

    n_components = min(5, U.shape[1])
    scores = U[:, :n_components] * S[:n_components]

    pca_scores_df = artifact_index_df.copy()
    for i in range(n_components):
        pca_scores_df[f"PC{i + 1}"] = scores[:, i]

    explained_df = pd.DataFrame({
        "component": [f"PC{i + 1}" for i in range(n_components)],
        "explained_variance": explained_var[:n_components],
        "explained_variance_ratio": explained_ratio[:n_components],
    })

    display(explained_df)
    display(pca_scores_df.head(20))

    if n_components >= 2:
        plt.figure(figsize=(8, 6))
        sns.scatterplot(
            data=pca_scores_df,
            x="PC1",
            y="PC2",
            hue="iteration",
            style="beam_sub_iteration",
            s=80,
        )
        plt.title("Weighted PCA Scores (PC1 vs PC2)")
        plt.tight_layout()


In [ ]:
# PCA exploratory diagnostics and loadings
if not all(name in globals() for name in ["X", "Vt", "pca_scores_df", "explained_df", "feature_index_df"]):
    print("PCA outputs missing. Run the PCA cell first.")
elif X.size == 0:
    print("PCA matrix is empty. Skipping diagnostics.")
else:
    explained_plot_df = explained_df.copy()
    explained_plot_df["cumulative_explained_variance_ratio"] = explained_plot_df["explained_variance_ratio"].cumsum()
    display(explained_plot_df)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.barplot(data=explained_plot_df, x="component", y="explained_variance_ratio", ax=axes[0], color="#4C72B0")
    axes[0].set_title("Explained Variance Ratio")
    axes[0].set_ylabel("ratio")
    axes[0].set_xlabel("component")

    sns.lineplot(data=explained_plot_df, x="component", y="cumulative_explained_variance_ratio", marker="o", ax=axes[1], color="#55A868")
    axes[1].set_title("Cumulative Explained Variance")
    axes[1].set_ylim(0, 1.01)
    axes[1].set_ylabel("cumulative ratio")
    axes[1].set_xlabel("component")
    plt.tight_layout()

    n_plot_components = min(3, Vt.shape[0])
    loadings_df = pd.DataFrame({"feature_index": np.arange(Vt.shape[1], dtype=int)})
    for i in range(n_plot_components):
        loadings_df[f"PC{i + 1}_loading"] = Vt[i, :]
        loadings_df[f"PC{i + 1}_abs_loading"] = np.abs(Vt[i, :])

    pca_feature_loadings_df = feature_index_df.merge(loadings_df, on="feature_index", how="left")
    pca_link_importance_df = (
        pca_feature_loadings_df
        .groupby("link", dropna=False)
        .agg(
            daily_volume=("daily_volume", "first"),
            cumulative_share=("cumulative_share", "first"),
            PC1_abs_loading_mean=("PC1_abs_loading", "mean"),
            PC1_abs_loading_max=("PC1_abs_loading", "max"),
            PC2_abs_loading_mean=("PC2_abs_loading", "mean") if "PC2_abs_loading" in pca_feature_loadings_df.columns else ("PC1_abs_loading", "mean"),
        )
        .reset_index()
        .sort_values("PC1_abs_loading_mean", ascending=False)
    )

    print("Top links by mean absolute PC1 loading:")
    display(pca_link_importance_df.head(20))

    hourly_profile = (
        pca_feature_loadings_df
        .groupby("hour_bin", dropna=False)
        .agg(PC1_abs_loading_mean=("PC1_abs_loading", "mean"))
        .reset_index()
        .sort_values("hour_bin")
    )
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=hourly_profile, x="hour_bin", y="PC1_abs_loading_mean", marker="o")
    plt.title("Mean Absolute PC1 Loading by Hour Bin")
    plt.xlabel("hour_bin")
    plt.ylabel("mean |loading|")
    plt.tight_layout()

    if pca_scores_df.shape[1] >= 9 and "phys_sim_iteration" in pca_scores_df.columns:
        plt.figure(figsize=(9, 4))
        sns.lineplot(
            data=pca_scores_df.sort_values(["iteration", "beam_sub_iteration", "phys_sim_iteration"], na_position="last"),
            x="phys_sim_iteration",
            y="PC1",
            hue="iteration",
            style="beam_sub_iteration",
            marker="o",
        )
        plt.title("PC1 Trajectory Across Phys-Sim Iterations")
        plt.tight_layout()


In [ ]:
# Enrich PCA loadings with BeamNetworkFinal link attributes
from pilates.database.schema.beam_schema import BeamNetworkFinal
from sqlmodel import Session, text

if "pca_link_importance_df" not in globals() or pca_link_importance_df.empty:
    print("No PCA link-importance table found. Run the PCA exploratory cell first.")
else:
    run_id_values = (
        summary_df.get("run_id", pd.Series(dtype=str))
        .dropna()
        .astype(str)
        .str.strip()
    )
    run_id_values = [v for v in run_id_values.unique().tolist() if v]
    run_id_for_network = run_id_values[0] if len(run_id_values) == 1 else None

    network_view_name = f"v_beam_network_final_y{TARGET_YEAR}"
    tracker.create_grouped_view(
        view_name=network_view_name,
        schema=BeamNetworkFinal,
        namespace="beam",
        attach_facets=["artifact_family", "year", "iteration"],
        include_system_columns=True,
        mode="hybrid",
        if_exists="replace",
        missing_files="skip_silent",
        year=int(TARGET_YEAR),
        run_id=run_id_for_network,
    )

    def q_ident(name: str) -> str:
        return '"' + str(name).replace('"', '""') + '"'

    with Session(tracker.engine) as session:
        info_rows = session.exec(text(f"PRAGMA table_info('{network_view_name}')")).all()
    colnames = [str(row[1]) for row in info_rows]

    def canon(name: str) -> str:
        return "".join(ch for ch in str(name).lower() if ch.isalnum())

    col_by_canon = {canon(c): c for c in colnames}

    def pick(*candidates):
        for c in candidates:
            resolved = col_by_canon.get(canon(c))
            if resolved is not None:
                return resolved
        return None

    link_col = pick("link_id", "linkId", "linkid")
    type_col = pick("attribute_orig_type", "attributeOrigType", "attributeorigtype")
    lanes_col = pick("number_of_lanes", "numberOfLanes", "numberoflanes")
    cap_col = pick("link_capacity", "linkCapacity", "linkcapacity")

    if link_col is None:
        raise ValueError(
            f"Could not find link id column in {network_view_name}; columns={colnames}"
        )

    select_parts = [f"CAST({q_ident(link_col)} AS BIGINT) AS link_id"]
    if type_col is not None:
        select_parts.append(f"ANY_VALUE({q_ident(type_col)}) AS attribute_orig_type")
    else:
        select_parts.append("NULL AS attribute_orig_type")
    if lanes_col is not None:
        select_parts.append(f"AVG(CAST({q_ident(lanes_col)} AS DOUBLE)) AS number_of_lanes")
    else:
        select_parts.append("NULL AS number_of_lanes")
    if cap_col is not None:
        select_parts.append(f"AVG(CAST({q_ident(cap_col)} AS DOUBLE)) AS link_capacity")
    else:
        select_parts.append("NULL AS link_capacity")

    sql = f"""
        SELECT
            {", ".join(select_parts)}
        FROM {q_ident(network_view_name)}
        WHERE {q_ident(link_col)} IS NOT NULL
        GROUP BY 1
    """
    with Session(tracker.engine) as session:
        network_rows = session.exec(text(sql)).all()

    network_link_df = pd.DataFrame(
        network_rows,
        columns=["link_id", "attribute_orig_type", "number_of_lanes", "link_capacity"],
    )
    network_link_df["link_id"] = pd.to_numeric(network_link_df["link_id"], errors="coerce").astype("Int64")

    pca_link_enriched_df = pca_link_importance_df.copy()
    pca_link_enriched_df["link_id"] = pd.to_numeric(
        pca_link_enriched_df["link"],
        errors="coerce",
    ).astype("Int64")
    pca_link_enriched_df = pca_link_enriched_df.merge(network_link_df, on="link_id", how="left")

    type_summary_df = (
        pca_link_enriched_df
        .groupby("attribute_orig_type", dropna=False)
        .agg(
            n_links=("link_id", "nunique"),
            mean_daily_volume=("daily_volume", "mean"),
            pc1_abs_loading_mean=("PC1_abs_loading_mean", "mean"),
            pc1_abs_loading_max=("PC1_abs_loading_max", "max"),
            mean_lanes=("number_of_lanes", "mean"),
            mean_capacity=("link_capacity", "mean"),
        )
        .reset_index()
        .sort_values("pc1_abs_loading_mean", ascending=False)
    )

    print("PCA link importance joined with network attributes:")
    display(pca_link_enriched_df.head(20))
    print("Functional-class summary (attribute_orig_type):")
    display(type_summary_df.head(20))

    top_types = type_summary_df.head(12).copy()
    if not top_types.empty:
        plt.figure(figsize=(10, 4))
        sns.barplot(
            data=top_types,
            x="attribute_orig_type",
            y="pc1_abs_loading_mean",
            color="#4C72B0",
        )
        plt.xticks(rotation=45, ha="right")
        plt.title("Mean |PC1 Loading| by attribute_orig_type")
        plt.tight_layout()


In [ ]:
# Deep-dive on PC2+ (hour profiles, class loadings, and metric correlations)
if not all(name in globals() for name in ["Vt", "feature_index_df", "pca_scores_df", "summary_df"]):
    print("Missing PCA artifacts. Run PCA cells first.")
else:
    n_eval = min(5, Vt.shape[0])
    if n_eval == 0:
        print("No components available.")
    else:
        eval_loadings = feature_index_df.copy()
        for i in range(n_eval):
            eval_loadings[f"PC{i+1}_loading"] = Vt[i, :]
            eval_loadings[f"PC{i+1}_abs_loading"] = np.abs(Vt[i, :])

        load_long = eval_loadings.melt(
            id_vars=["feature_index", "link", "hour_bin"],
            value_vars=[f"PC{i+1}_abs_loading" for i in range(n_eval)],
            var_name="pc",
            value_name="abs_loading",
        )
        load_long["pc"] = load_long["pc"].str.replace("_abs_loading", "", regex=False)

        hour_pc = (
            load_long
            .groupby(["hour_bin", "pc"], dropna=False)["abs_loading"]
            .mean()
            .reset_index()
        )
        hour_pc_pivot = hour_pc.pivot(index="hour_bin", columns="pc", values="abs_loading")
        plt.figure(figsize=(8, 5))
        sns.heatmap(hour_pc_pivot, cmap="viridis")
        plt.title("Mean Absolute Loading by Hour Bin and Component")
        plt.tight_layout()

        if "pca_link_enriched_df" in globals() and not pca_link_enriched_df.empty and "attribute_orig_type" in pca_link_enriched_df.columns:
            type_map = pca_link_enriched_df[["link", "attribute_orig_type"]].drop_duplicates(subset=["link"], keep="first")
            class_load = (
                load_long.merge(type_map, on="link", how="left")
                .groupby(["pc", "attribute_orig_type"], dropna=False)
                .agg(
                    mean_abs_loading=("abs_loading", "mean"),
                    max_abs_loading=("abs_loading", "max"),
                    n_features=("feature_index", "count"),
                )
                .reset_index()
            )
            top_class_load = (
                class_load.sort_values(["pc", "mean_abs_loading"], ascending=[True, False])
                .groupby("pc", as_index=False)
                .head(8)
            )
            display(top_class_load)
            g = sns.catplot(
                data=top_class_load,
                x="attribute_orig_type",
                y="mean_abs_loading",
                col="pc",
                kind="bar",
                col_wrap=3,
                sharey=False,
                height=3.2,
                aspect=1.3,
            )
            g.set_xticklabels(rotation=45, ha="right")
            g.fig.subplots_adjust(top=0.9)
            g.fig.suptitle("Top Functional Classes by Mean Absolute Loading")

        score_metric_df = pca_scores_df.merge(
            summary_df[[
                "artifact_id",
                "vmt_miles",
                "vht_hours",
                "total_delay_hours",
                "traveltime_mean",
                "volume_sum",
            ]],
            on="artifact_id",
            how="left",
        )
        pc_cols = [f"PC{i+1}" for i in range(n_eval)]
        metric_cols = [
            "vmt_miles",
            "vht_hours",
            "total_delay_hours",
            "traveltime_mean",
            "volume_sum",
        ]
        corr_df = score_metric_df[pc_cols + metric_cols].corr().loc[pc_cols, metric_cols]
        display(corr_df)
        plt.figure(figsize=(8, 4))
        sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", center=0)
        plt.title("Correlation: PC Scores vs Linkstats Metrics")
        plt.tight_layout()


In [ ]:
# PC systemwide-vs-localized diagnostics (sign coherence + concentration)
if not all(name in globals() for name in ["Vt", "feature_index_df"]):
    print("Missing PCA loadings. Run PCA cells first.")
else:
    n_eval = min(5, Vt.shape[0])
    if n_eval == 0:
        print("No components available.")
    else:
        feature_diag = feature_index_df[["feature_index", "link", "hour_bin", "daily_volume"]].copy()
        feature_diag["weight"] = pd.to_numeric(feature_diag["daily_volume"], errors="coerce").fillna(0.0).clip(lower=0.0)
        if float(feature_diag["weight"].sum()) <= 0:
            feature_diag["weight"] = 1.0

        for i in range(n_eval):
            feature_diag[f"PC{i+1}_loading"] = Vt[i, :]

        if "pca_link_enriched_df" in globals() and not pca_link_enriched_df.empty and "attribute_orig_type" in pca_link_enriched_df.columns:
            class_map = pca_link_enriched_df[["link", "attribute_orig_type"]].drop_duplicates(subset=["link"], keep="first")
            class_map["attribute_orig_type"] = class_map["attribute_orig_type"].fillna("<missing>").astype(str)
        else:
            class_map = pd.DataFrame({"link": feature_diag["link"].astype(str).unique(), "attribute_orig_type": "<unknown>"})

        feature_diag["link"] = feature_diag["link"].astype(str)
        class_map["link"] = class_map["link"].astype(str)
        feature_diag = feature_diag.merge(class_map, on="link", how="left")
        feature_diag["attribute_orig_type"] = feature_diag["attribute_orig_type"].fillna("<missing>")

        pc_localization_rows = []
        pc_class_rows = []

        for i in range(n_eval):
            pc = f"PC{i+1}"
            vals = pd.to_numeric(feature_diag[f"{pc}_loading"], errors="coerce").fillna(0.0).to_numpy()
            w = feature_diag["weight"].to_numpy(dtype=float)

            orient = np.sign(np.sum(w * vals))
            if orient == 0:
                orient = 1.0
            vals = vals * orient

            mass = w * np.abs(vals)
            total_mass = float(mass.sum())
            if total_mass <= 0:
                continue

            p = mass / total_mass
            hhi = float(np.sum(p ** 2))
            n_eff = float(1.0 / hhi) if hhi > 0 else float("nan")
            n_features = len(p)

            def top_share(frac: float) -> float:
                k = max(1, int(np.ceil(n_features * frac)))
                return float(np.sort(p)[::-1][:k].sum())

            pc_localization_rows.append({
                "pc": pc,
                "hhi": hhi,
                "n_eff": n_eff,
                "top_1pct_share": top_share(0.01),
                "top_5pct_share": top_share(0.05),
                "top_10pct_share": top_share(0.10),
                "n_features": n_features,
            })

            tmp = feature_diag[["attribute_orig_type", "weight"]].copy()
            tmp["loading"] = vals
            tmp["abs_mass"] = tmp["weight"] * np.abs(tmp["loading"])
            tmp["signed_mass"] = tmp["weight"] * tmp["loading"]
            tmp["pos_mass"] = tmp["weight"] * np.clip(tmp["loading"], 0.0, None)
            tmp["neg_mass"] = tmp["weight"] * np.clip(-tmp["loading"], 0.0, None)

            class_agg = (
                tmp.groupby("attribute_orig_type", dropna=False)
                .agg(
                    abs_mass=("abs_mass", "sum"),
                    signed_mass=("signed_mass", "sum"),
                    pos_mass=("pos_mass", "sum"),
                    neg_mass=("neg_mass", "sum"),
                    feature_count=("loading", "count"),
                )
                .reset_index()
            )
            class_agg["pc"] = pc
            class_agg["sign_coherence"] = np.where(
                class_agg["abs_mass"] > 0,
                np.abs(class_agg["signed_mass"]) / class_agg["abs_mass"],
                np.nan,
            )
            class_agg["sign_balance"] = np.where(
                class_agg["abs_mass"] > 0,
                class_agg["signed_mass"] / class_agg["abs_mass"],
                np.nan,
            )
            class_agg["positive_mass_share"] = np.where(
                (class_agg["pos_mass"] + class_agg["neg_mass"]) > 0,
                class_agg["pos_mass"] / (class_agg["pos_mass"] + class_agg["neg_mass"]),
                np.nan,
            )
            class_agg["within_pc_mass_share"] = class_agg["abs_mass"] / total_mass
            pc_class_rows.append(class_agg)

        pc_localization_df = pd.DataFrame(pc_localization_rows)
        pc_class_diagnostics_df = pd.concat(pc_class_rows, ignore_index=True) if pc_class_rows else pd.DataFrame()

        display(pc_localization_df)
        if not pc_class_diagnostics_df.empty:
            top_class_diag = (
                pc_class_diagnostics_df
                .sort_values(["pc", "within_pc_mass_share"], ascending=[True, False])
                .groupby("pc", as_index=False)
                .head(8)
            )
            display(top_class_diag[[
                "pc",
                "attribute_orig_type",
                "within_pc_mass_share",
                "sign_coherence",
                "sign_balance",
                "positive_mass_share",
            ]])

            coh_pivot = (
                top_class_diag
                .pivot(index="attribute_orig_type", columns="pc", values="sign_coherence")
                .fillna(0.0)
            )
            plt.figure(figsize=(8, 4))
            sns.heatmap(coh_pivot, annot=True, fmt=".2f", cmap="mako", vmin=0, vmax=1)
            plt.title("Class Sign Coherence by Principal Component")
            plt.tight_layout()

        if not pc_localization_df.empty:
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            sns.barplot(data=pc_localization_df, x="pc", y="n_eff", ax=axes[0], color="#4C72B0")
            axes[0].set_title("Effective Number of Active Features (N_eff)")
            axes[0].set_ylabel("N_eff")

            top_share_plot = pc_localization_df.melt(
                id_vars=["pc"],
                value_vars=["top_1pct_share", "top_5pct_share", "top_10pct_share"],
                var_name="bucket",
                value_name="share",
            )
            sns.barplot(data=top_share_plot, x="pc", y="share", hue="bucket", ax=axes[1])
            axes[1].set_title("Concentration of Absolute Loading Mass")
            axes[1].set_ylabel("share")
            axes[1].set_ylim(0, 1)
            plt.tight_layout()


## Notes

- This notebook intentionally mounts `workspace://` to the archived run directory.
- It computes metrics from a single Consist grouped hybrid view (`tracker.create_grouped_view(...)`) rather than direct parquet paths.
- If view creation fails, verify mounts and ensure your Consist DB schema matches your Consist package version.
- You can switch artifact family to `linkstats_parquet` or `linkstats` in `find_linkstats_artifacts(...)` for other analyses.
